## File: `timeit_Algo-2_Algo-3.ipynb`  
**Project:** HHG-SaDAS  

### Code Description  
This notebook measures the CPU time required to execute the code.  

Following Appendix-A of my thesis:  
- **Algo-2** also uses an equispaced grid in $x \in (-1, 1)$. However, unlike Algo-1,  
  it estimates the roots of $\Lambda_N(x)$, requiring a much smaller grid size to generate  
  the initial guesses.
  
- **Algo-3** uses the non-equispaced grid $x^+(\xi) \in (0, 1)$, as in Eq.A.11,  
  to determine the roots of $\Lambda_N(x)$ from the local maxima of $-(\Lambda_N(x))^2$ (serving as the initial guesses).  
  It calculates only half of the roots (those in the positive interval), while the other half  
  (in the negative interval) are obtained using the parity relation in Eq.A.10.  
  This algorithm is graphically presented in the flowchart of Fig.A.4.
  
- It does not save data; rather, it shows detailed execution time analysis.

---

**Author:** Siddhartha Mithiya  
**Affiliation:** Indian Institute of Technology (IIT) Mandi  
**License:** MIT License  
**Repository:** [https://github.com/Siddhartha-Acad/HHG-SaDAS.git](https://github.com/Siddhartha-Acad/HHG-SaDAS.git)  

---

### Notes  
- Collocation points are calculated using the zeros of $P'_N$ (analytical derivative) with the `fsolve` function of SciPy.  
- This notebook is part of the HHG-SaDAS package, developed during my MS(R) thesis:  
  *"Higher-Order Harmonic Generation and Harmonic-Power Enhancement in Noble-Gas Atoms Confined Inside C60".*


In [1]:
import timeit
import warnings
import numpy as np
from scipy.optimize import fsolve
from scipy.special import legendre
from scipy.signal import find_peaks

In [2]:
def P_N_deriv(x, N):
    return legendre(N-1)(x) - legendre(N+1)(x)


## [Algo-2]: x array is simple equispaced array with large grid size

In [3]:
N = 5
nopx = 1000
x = np.linspace(-1, 1, nopx)

PN_deriv_array = P_N_deriv(x, N)
pks_at = find_peaks(-PN_deriv_array ** 2)[0]
colloc_pt = np.array([fsolve(P_N_deriv, x[pks_at[j]], args=(N,))[0] for j in range(len(pks_at))])

# Raise a warning if the number of collocation points is less than expected
expected_points = N - 1
if len(colloc_pt) < expected_points:
    print(f"⚠️ Warning: Grid size (nopx={nopx}) is not sufficient: found {len(colloc_pt)}, expected {expected_points}.")
    print("   Consider increasing 'nopx' to resolve all initial guesses.")


In [4]:

errors = P_N_deriv(colloc_pt, N)
mean_error = np.mean(errors)
std_dev_error = np.std(errors, ddof=1)  # Using ddof=1 for sample standard deviation

for i in range(len(colloc_pt)):
    print(f'xj[{i}]= {colloc_pt[i]:.5f}; err= {errors[i]:.5e}')


max_abs_value = max(abs(mean_error), abs(std_dev_error))
power = int(np.floor(np.log10(max_abs_value)))

scaled_mean = mean_error / 10**power
scaled_std_dev = std_dev_error / 10**power

print(f"\n# colloc points (N={N})   : {len(colloc_pt)}")
print(f"grid info: xi, xf, nopx  : {x[0], x[-1], nopx}")
print(f"err: mean ± standard dev : ({scaled_mean:.2f} ± {scaled_std_dev:.2f}) × 10^{power}")


xj[0]= -0.76506; err= -4.44089e-16
xj[1]= -0.28523; err= -2.22045e-16
xj[2]= 0.28523; err= -5.55112e-17
xj[3]= 0.76506; err= 2.22045e-16

# colloc points (N=5)   : 4
grid info: xi, xf, nopx  : (-1.0, 1.0, 1000)
err: mean ± standard dev : (-1.25 ± 2.81) × 10^-16


### Time profile

In [5]:
%%timeit
x = np.linspace(-1, 1, nopx)

PN_deriv_array = P_N_deriv(x, N)
pks_at = find_peaks(-PN_deriv_array ** 2)[0]
colloc_pt = np.array([fsolve(P_N_deriv, x[pks_at[j]], args=(N,))[0] for j in range(len(pks_at))])


8.68 ms ± 1.12 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


## [Algo-3]: x array is mapped from 0 to 1
- Only half of the roots are calculated and rest are done by parity.

In [6]:
def f_rev(x_array):             # f(x) reversed to have dense grid towards far.
    r_max = 1; L_map = 0.2
    alpha = 2 * L_map / r_max
    map_func = L_map*(1 + x_array) / (1 - x_array + alpha)
    return -map_func[::-1] + r_max


In [7]:
nopx = int(nopx/2)
xi = np.linspace(-1, 1, nopx)
x_mapped = f_rev(xi)

PN_deriv_array = P_N_deriv(x_mapped, N)
pks_at = find_peaks(-PN_deriv_array ** 2)[0]
colloc_pt = np.array([fsolve(P_N_deriv, x_mapped[pks_at[j]], args=(N,))[0] for j in range(len(pks_at))])

if N % 2 == 0:
    colloc_pt = np.concatenate((-colloc_pt[::-1], [0.0], colloc_pt))
else:
    colloc_pt = np.concatenate((-colloc_pt[::-1], colloc_pt))
assert len(colloc_pt) == N-1, f"Grid size (nopx) is not enough to resolve {N-1} initial guesses"


In [8]:
errors = P_N_deriv(colloc_pt, N)
mean_error = np.mean(errors)
std_dev_error = np.std(errors, ddof=1)  # Using ddof=1 for sample standard deviation

for i in range(N-1):
    print(f'xj[{i}]= {colloc_pt[i]:.5f}; err= {errors[i]:.5e}')

max_abs_value = max(abs(mean_error), abs(std_dev_error))
power = int(np.floor(np.log10(max_abs_value)))

scaled_mean = mean_error / 10**power
scaled_std_dev = std_dev_error / 10**power

print(f"\n# colloc points (N={N})   : {len(colloc_pt)}")
print(f"grid info: xi, xf, nopx  : {x[0], x[-1], nopx}")
print(f"err: mean ± standard dev : ({scaled_mean:.2f} ± {scaled_std_dev:.2f}) × 10^{power}")


xj[0]= -0.76506; err= 3.33067e-16
xj[1]= -0.28523; err= -1.11022e-16
xj[2]= 0.28523; err= -1.11022e-16
xj[3]= 0.76506; err= 2.22045e-16

# colloc points (N=5)   : 4
grid info: xi, xf, nopx  : (-1.0, 1.0, 500)
err: mean ± standard dev : (0.83 ± 2.29) × 10^-16


### Time profile

In [9]:
%%timeit
x_mapped = f_rev(xi)

PN_deriv_array = P_N_deriv(x_mapped, N)
pks_at = find_peaks(-PN_deriv_array ** 2)[0]
colloc_pt = np.array([fsolve(P_N_deriv, x_mapped[pks_at[j]], args=(N,))[0] for j in range(len(pks_at))])


6.06 ms ± 328 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
